In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

import dowhy
from dowhy import CausalModel

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import graphviz

Suppose you are considering the following DAG, where $T$ is the treatment node and $Y$ is the outcome node:

In [ ]:
graph = graphviz.Digraph(format='png', engine='neato')

nodes = ['S', 'Q', 'T', 'Y', 'P']
positions = ['2,2.5!', '3,1!', '3,0!', '1, 0!', '1,2!', '1.7,0.9!']

edges = ['SQ', 'SY', 'QT', 'QY', 'TP', 'YP', 'TY']

[graph.node(n, pos=pos) for n, pos in zip(nodes, positions)]
graph.edges(edges)

graph.render(f'img/ch_07_full_example')

graph

**Question.** How many confounders are there between $T$ and $Y$? What should you control for to make the causal effect of $T$ over $Y$ identifiable?

We are going to generate $N=1000$ samples from this graphicam model:

In [ ]:
N = 1000

S = np.random.random(N)
Q = 0.2*S + 0.67*np.random.random(N)
T = 0.14*Q + 0.4*np.random.random(N)
Y = 0.7*T + 0.11*Q + 0.32*S + 0.24*np.random.random(N)
P = 0.43*T + 0.21*Y + 0.22*np.random.random(N)

In [ ]:
# Encode as a pandas df
df = pd.DataFrame(np.vstack([S, Q, T, Y, P]).T, columns=['S', 'Q', 'T', 'Y', 'P'])
df

We instantiate the causal model in DoWhy:

In [ ]:
# Generate the GML graph
gml_string = 'graph [directed 1\n'

for node in nodes:
    gml_string += f'\tnode [id "{node}" label "{node}"]\n'

for edge in edges:
    gml_string += f'\tedge [source "{edge[0]}" target "{edge[1]}"]\n'
    
gml_string += ']'

In [ ]:
print(gml_string)

In [ ]:
model = CausalModel(
    data=df,
    treatment='T',
    outcome='Y',
    graph=gml_string
)

And we visualize it:

In [ ]:
model.view_model()

DoWhy can automatically help finding the good estimand:

In [ ]:
# Get the estimand
estimand = model.identify_effect()
print(estimand)

DoWhy proposed one valid estimand: backdoor. Indeed, this model contains only one back-door path: controlling for $Q$ deconfounds the relationship between $T$ and $Y$.

We can now proceed to get an estimation. Although the true SCM is simple and linear regression could be enough, we can use a more advanced estimator, such as `LassoCV` and `GradientBoostingRegressor`.

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.ensemble import GradientBoostingRegressor

We can use `DML` estimator from the `EconML` package: it fits 3 MLg models to compute de-biased estimates of treatment effects. First, it predicts the outcome from the controls. Then, it predicts the treatment from the controls. Finally, it fits the model that regresses residuals from the second model on the residuals from the first model.

In [ ]:
estimate = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.econml.dml.DML',
    method_params={
        'init_params': {
            'model_y': GradientBoostingRegressor(),
            'model_t': GradientBoostingRegressor(),
            'model_final': LassoCV(fit_intercept=False),
        },
        'fit_params': {}}
)

print(f'Estimate of causal effect (DML): {estimate.value}')

In [ ]:
estimate_lr = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.linear_regression')

print(f'Estimate of causal effect (linear regression): {estimate_lr.value}')